In [1]:
# Proximal Policy Optimization (PPO)
#
# https://spinningup.openai.com/en/latest/algorithms/ppo.html
# https://docs.pytorch.org/rl/stable/tutorials/coding_ppo.html

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
import torch
from tensordict.nn import NormalParamExtractor, TensorDictModule
from torch import nn
from torchrl import logger
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, DoubleToFloat, GymEnv, StepCounter, TransformedEnv
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

_ = torch.manual_seed(0)

In [ ]:
env = TransformedEnv(
    GymEnv("InvertedPendulum-v5"),
    Compose([DoubleToFloat(), StepCounter()]),
)

_ = env.set_seed(0)

In [5]:
NUM_CELLS = 128

# PPO uses a stochastic policy which is determined by the parameters (loc and scale) of
# a Normal distribution (for continuous action spaces).

# maps state to parameters (loc, scale) of a Normal distribution
policy_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(2),
    NormalParamExtractor(),
)

# stochastic policy which samples actions from the TanhNormal distribution
policy_module = ProbabilisticActor(
    module=TensorDictModule(
        policy_net, in_keys=["observation"], out_keys=["loc", "scale"]
    ),
    spec=env.action_spec,
    in_keys=["loc", "scale"],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec.space.low.item(),  # type: ignore
        "high": env.action_spec.space.high.item(),  # type: ignore
    },
    return_log_prob=True,
)

In [ ]:
value_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(1),
)

# TensorDictModule with in_keys=["observation"] and out_keys=["state_value"]
value_module = ValueOperator(value_net)

In [7]:
_ = value_module(env.rollout(5, policy_module))  # initialize lazy layers

In [ ]:
LEARNING_RATE = 1e-4

advantage_module = GAE(
    gamma=0.99,
    lmbda=0.95,
    value_network=value_module,
    average_gae=True,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=0.2,
    entropy_bonus=False,
)

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 10_000)

In [9]:
FRAMES_PER_BATCH = 100

collector = Collector(
    env,
    policy=policy_module,
    frames_per_batch=FRAMES_PER_BATCH,
    total_frames=-1,
)

# Buffer to store a single batch of data from the collector.
# The replay buffer is only used for sub-sampling when splitting
# a batch of data into minibatches; it only contains the most recent
# data collected using the current policy (on-policy learning).
buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=FRAMES_PER_BATCH))

In [ ]:
# Evaluate the policy in a deterministic manner for one episode.
# Instead of sampling, the policy selects the best action at each step.


@torch.no_grad()
def evaluate_policy(env, policy_module: ProbabilisticActor):
    env.reset()

    with set_exploration_type(ExplorationType.DETERMINISTIC):
        rollout = env.rollout(1000, policy_module)

    max_steps = rollout["next", "step_count"].max().item()
    env.reset()

    return max_steps

In [ ]:
NUM_EPOCHS = 10
MINI_BATCH_SIZE = 64


step_count = 0
episode_count = 0

for idx, data in enumerate(collector, start=1):
    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    max_steps = evaluate_policy(env, policy_module)

    if max_steps > 200:
        break

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] max: {max_steps:>3}")

    # train on the entire batch multiple times
    for _ in range(NUM_EPOCHS):
        advantage_module(data)  # recompute GAE as it depends on the value module
        buffer.extend(data)  # replace previous data

        # partition batch into minibatches and take one optimizer step per minibatch
        for _ in range(FRAMES_PER_BATCH // MINI_BATCH_SIZE):
            mini_batch = buffer.sample(MINI_BATCH_SIZE)
            loss_values = loss_module(mini_batch)

            loss = loss_values["loss_objective"] + loss_values["loss_critic"]
            loss.backward()

            nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

    scheduler.step()


logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-15 16:25:04,121 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100]) shape [END]
2026-02-15 16:25:06,957 [torchrl][INFO]    [ 10] max:  20 [END]
2026-02-15 16:25:10,199 [torchrl][INFO]    [ 20] max:  76 [END]
2026-02-15 16:25:14,186 [torchrl][INFO]    [ 30] max: 127 [END]
2026-02-15 16:25:16,230 [torchrl][INFO]    solved after 3400 steps, 182 episodes [END]
